## **EyeGaze "Fixation" Data Processing**

In [ ]:
import os
import pandas as pd
import numpy as np

# Root MIMIC-Eye path
root_path = r"mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\\mimic-eye"

for patient_folder in os.listdir(root_path):
    patient_path = os.path.join(root_path, patient_folder)

    if os.path.isdir(patient_path) and patient_folder.startswith("patient_"):
        eyegaze_path = os.path.join(patient_path, "EyeGaze")
        cxr_jpg_path = os.path.join(patient_path, "CXR-JPG")
        cxr_meta_path = os.path.join(cxr_jpg_path, "cxr_meta.csv")
        fixations_path = os.path.join(eyegaze_path, "fixations.csv")
        output_csv_path = os.path.join(eyegaze_path, "norm_fixations.csv")

        if os.path.exists(eyegaze_path) and os.path.exists(cxr_meta_path) and os.path.exists(fixations_path):
            try:
                cxr_df = pd.read_csv(cxr_meta_path)
                fix_df = pd.read_csv(fixations_path)

                # Compute baseline pupil area from first 2 seconds of valid data
                valid_baseline = fix_df[(fix_df['LPV'] == 1) & (fix_df['RPV'] == 1)]
                baseline_window = valid_baseline[valid_baseline['Time (in secs)'] <= 2.0]

                if not baseline_window.empty:
                    left_area = np.pi * (baseline_window['LPD'] / 2) ** 2
                    right_area = np.pi * (baseline_window['RPD'] / 2) ** 2
                    avg_area = (left_area + right_area) / 2
                    baseline_avg_area = avg_area.mean()
                else:
                    baseline_avg_area = None

                required_meta_cols = {'study_id', 'dicom_id'}
                required_fix_cols = {'DICOM_ID', 'FPOGX', 'FPOGY', 'FPOGD', 'LPD', 'RPD'}

                if required_meta_cols.issubset(cxr_df.columns) and required_fix_cols.issubset(fix_df.columns):
                    local_rows = []

                    for _, row in cxr_df.iterrows():
                        study_id = row['study_id']
                        dicom_id = row['dicom_id']
                        study_folder_path = os.path.join(cxr_jpg_path, f"s{study_id}")
                        image_path = os.path.join(study_folder_path, f"{dicom_id}.jpg")

                        if os.path.exists(image_path):
                            fix_matches = fix_df[fix_df['DICOM_ID'] == dicom_id]

                            if not fix_matches.empty:
                                for _, fix in fix_matches.iterrows():
                                    pupil_area_norm = ""
                                    if baseline_avg_area and fix['LPD'] > 0 and fix['RPD'] > 0:
                                        left_area = np.pi * (fix['LPD'] / 2) ** 2
                                        right_area = np.pi * (fix['RPD'] / 2) ** 2
                                        pupil_avg = (left_area + right_area) / 2
                                        pupil_area_norm = pupil_avg / baseline_avg_area

                                    local_rows.append({
                                        "study_id": study_id,
                                        "dicom_id": dicom_id,
                                        "x_norm": fix["FPOGX"],
                                        "y_norm": fix["FPOGY"],
                                        "gaze_duration": fix["FPOGD"],
                                        "pupil_area_normalized": pupil_area_norm
                                    })
                    # Save per patient
                    if local_rows:
                        df_local = pd.DataFrame(local_rows)
                        df_local.to_csv(output_csv_path, index=False)
                        print(f"Saved {len(df_local)} rows to {output_csv_path}")
                    else:
                        print(f"No valid fixation data for {patient_folder}")
                else:
                    print(f"Missing required columns in: {cxr_meta_path} or {fixations_path}")
            except Exception as e:
                print(f"Error in patient {patient_folder}: {e}")


## **REFLACX "Fixations" Data Processing**

In [ ]:
import os
import pandas as pd

# Root dataset path
root_path = r"mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\mimic-eye"

for patient_folder in os.listdir(root_path):
    patient_path = os.path.join(root_path, patient_folder)

    if not patient_folder.startswith("patient_"):
        continue

    reflacx_main_data_path = os.path.join(patient_path, "REFLACX", "main_data")
    cxr_meta_path = os.path.join(patient_path, "CXR-JPG", "cxr_meta.csv")
    output_csv_path = os.path.join(patient_path, "REFLACX", "norm_fixations.csv")

    if not os.path.exists(reflacx_main_data_path) or not os.path.exists(cxr_meta_path):
        continue

    try:
        cxr_df = pd.read_csv(cxr_meta_path)
        if not {'study_id', 'dicom_id'}.issubset(cxr_df.columns):
            print(f"Missing study_id/dicom_id in {cxr_meta_path}")
            continue

        rows = []

        for reflacx_id in os.listdir(reflacx_main_data_path):
            fix_path = os.path.join(reflacx_main_data_path, reflacx_id, "fixations.csv")
            if not os.path.exists(fix_path):
                continue

            fix_df = pd.read_csv(fix_path)

            required_cols = {
                'x_position', 'y_position',
                'xmin_shown_from_image', 'xmax_shown_from_image',
                'ymin_shown_from_image', 'ymax_shown_from_image',
                'pupil_area_normalized',
                'timestamp_start_fixation', 'timestamp_end_fixation'
            }
            if not required_cols.issubset(fix_df.columns):
                print(f"Required columns missing in {fix_path}")
                continue

            # Compute normalized coordinates
            fix_df['x_norm'] = (
                (fix_df['x_position'] - fix_df['xmin_shown_from_image']) /
                (fix_df['xmax_shown_from_image'] - fix_df['xmin_shown_from_image'])
            ).clip(0, 1)

            fix_df['y_norm'] = (
                (fix_df['y_position'] - fix_df['ymin_shown_from_image']) /
                (fix_df['ymax_shown_from_image'] - fix_df['ymin_shown_from_image'])
            ).clip(0, 1)

            # Compute gaze duration
            fix_df['gaze_duration'] = (
                fix_df['timestamp_end_fixation'] - fix_df['timestamp_start_fixation']
            )

            for _, fix in fix_df.iterrows():
                rows.append({
                    "study_id": study_id,
                    "dicom_id": dicom_id,
                    "reflacx_id": reflacx_id,
                    "x_norm": fix["x_norm"],
                    "y_norm": fix["y_norm"],
                    "gaze_duration": fix["gaze_duration"],
                    "pupil_area_normalized": fix["pupil_area_normalized"]
                })

        if rows:
            pd.DataFrame(rows).to_csv(output_csv_path, index=False)
            print(f"Saved {len(rows)} rows to {output_csv_path}")
        else:
            print(f"No fixations found for {patient_folder}")

    except Exception as e:
        print(f"Error in {patient_folder}: {e}")

## **Bounding Box for REFLACX**

In [ ]:
import os
import csv
import pandas as pd

# Root directory
root_path = r"mimic-eye-integrating-mimic-datasets-with-reflacx-and-eye-gaze-for-multimodal-deep-learning-applications-1.0.0\\mimic-eye"

# Destination CSV file
csv_output_path = r"patients_with_reflacx_folder.csv"

# To store results
rows = []

# Loop over all patient folders
for patient_folder in os.listdir(root_path):
    patient_path = os.path.join(root_path, patient_folder)

    if os.path.isdir(patient_path) and patient_folder.startswith("patient_"):
        reflacx_path = os.path.join(patient_path, "REFLACX", "main_data")
        cxr_jpg_path = os.path.join(patient_path, "CXR-JPG")
        cxr_meta_path = os.path.join(cxr_jpg_path, "cxr_meta.csv")
        
        # Check if REFLACX and cxr_meta.csv exist
        if os.path.exists(reflacx_path) and os.path.exists(cxr_meta_path):
            try:
                df = pd.read_csv(cxr_meta_path)
                if {'study_id', 'dicom_id'}.issubset(df.columns):
                    for _, row in df.iterrows():
                        study_id = row['study_id']
                        dicom_id = row['dicom_id']
                        study_folder_path = os.path.join(cxr_jpg_path, f"s{study_id}")

                        # Image path
                        image_path = os.path.join(study_folder_path, f"{dicom_id}.jpg")
                        
                        # Check if image file exists
                        if os.path.exists(image_path):
                            rows.append({
                                "patient_id": patient_folder,
                                "study_id": study_id,
                                "dicom_id": dicom_id,
                                "image_path": image_path,
                                "reflacx_main_folder_path": reflacx_path
                            })
                        else:
                            print(f"Image missing: {image_path}")
                else:
                    print(f"'study_id' or 'dicom_id' missing in {cxr_meta_path}")
            except Exception as e:
                print(f"Error reading {cxr_meta_path}: {e}")

# Write to CSV
with open(csv_output_path, mode='w', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=[
        "patient_id", "study_id", "dicom_id", "image_path", "reflacx_main_folder_path"
    ])
    writer.writeheader()
    writer.writerows(rows)

print(f"Successfully saved {len(rows)} rows into '{csv_output_path}'")

In [1]:
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import os

# --- CONFIG ---
reflacx_csv = r"patients_with_reflacx_folder.csv"
weights_path = r"best.pt"

# --- Load Model ---
model = YOLO(weights_path)

# --- Load Patient Image Metadata ---
df = pd.read_csv(reflacx_csv)

# --- Storage ---
patient_bboxes = {}

# --- Predict per Image ---
for idx, row in df.iterrows():
    patient_id = row['patient_id']
    dicom_id = row['dicom_id']
    study_id = row['study_id']
    image_path = row['image_path']
    reflacx_folder = os.path.join(Path(image_path).parents[2], "REFLACX")

    try:
        preds = model.predict(
            source=image_path,
            save=False,
            save_txt=False,
            conf=0.05,
            iou=0.5,
            device=0,
            verbose=False
        )[0]

        # Reset best boxes for this image
        best_per_class = {}

        for box in preds.boxes:
            cls_id = int(box.cls.item())
            conf = float(box.conf.item())
            x1, y1, x2, y2 = box.xyxy[0].tolist()

            if cls_id not in best_per_class or best_per_class[cls_id]["confidence"] < conf:
                best_per_class[cls_id] = {
                    "study_id": study_id,
                    "dicom_id": dicom_id,
                    "class_id": cls_id,
                    "class_name": model.names[cls_id],
                    "confidence": conf,
                    "x1": x1,
                    "y1": y1,
                    "x2": x2,
                    "y2": y2
                }

        if patient_id not in patient_bboxes:
            patient_bboxes[patient_id] = {
                "reflacx_folder": reflacx_folder,
                "entries": []
            }

        patient_bboxes[patient_id]["entries"].extend(best_per_class.values())
        print(f"{dicom_id}: {len(best_per_class)} unique class boxes saved")

    except Exception as e:
        print(f"Error on {image_path}: {e}")

# --- Save per-patient CSVs with safe checks ---
for patient_id, data in patient_bboxes.items():
    reflacx_folder = data["reflacx_folder"]
    save_path = os.path.join(reflacx_folder, "reflacx_bounding_boxes.csv")
    os.makedirs(reflacx_folder, exist_ok=True)

    df_all = pd.DataFrame(data["entries"])
    if df_all.empty or 'confidence' not in df_all.columns:
        print(f"Skipped empty or invalid data for patient: {patient_id}")
        continue

    # **Keep only one box per class_id, the one with highest confidence**
        # Group by each image (dicom_id), then for each image
    for dicom_id, df_img in df_all.groupby("dicom_id"):
        # keep only the highest-confidence box per class in that image
        df_best = (
            df_img
            .sort_values("confidence", ascending=False)
            .drop_duplicates(subset=["class_id"], keep="first")
            .sort_values(by=["class_id"])
        )

        # save under REFLACX/<dicom_id>_bboxes.csv
        img_csv = os.path.join(reflacx_folder, f"{dicom_id}_bboxes.csv")
        df_best.to_csv(img_csv, index=False)
        print(f"Saved {len(df_best)} boxes for image {dicom_id} → {img_csv}")



print("\nAll patient CSVs now safely filtered with unique [dicom_id, class_id].")

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\tanji\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


d:\Studies\Projects\Mimic\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


fa771fa1-d9571d07-bff8f655-327734a7-6e10b29d: 15 unique class boxes saved
dcdc4bd9-4301b111-2a65a814-ee8e7bc5-7f0b9a5a: 1 unique class boxes saved
5bdabba9-388f6646-ac06b5f5-f68b2fd2-3630de21: 5 unique class boxes saved
4a629500-9c3281ca-90bab490-9b6ac9c1-e5e6a580: 0 unique class boxes saved
c48dc8db-35b347c7-deafe540-9d161fdd-7d88173a: 14 unique class boxes saved
38a5ffc8-93297f03-3f0a14e4-aa8ca225-7e968b5b: 17 unique class boxes saved
014cf120-d578617b-edc3682a-28dc6244-d719eda5: 2 unique class boxes saved
fbae5604-a1013966-9879d4a3-7298599f-956824e1: 1 unique class boxes saved
8ccf0b84-84013d2f-287bdfe0-510bfd7b-865dd733: 8 unique class boxes saved
f7b69ee3-db7f264c-fca7d1c7-1d372fc0-02b35a47: 17 unique class boxes saved
d98be0d5-a648e485-4473a3ef-7762b3ef-2d70507e: 10 unique class boxes saved
56b4e225-4dfb070d-783280b4-da33f762-168aa398: 1 unique class boxes saved
6c79a3c1-8d01e0d4-b8b42695-900cae5d-18a8e87a: 17 unique class boxes saved
abea5eb9-b7c32823-3a14c5ca-77868030-69c83139: